# EdgentRAG model services on Lightning AI

This notebook runs both the embedding API (port 8001) and generation API (port 8003) in the current Lightning Studio. It uses Lightning's SDK to obtain the exposed port URLs, then verifies each URL using an authenticated API request.

Set `EDGENTRAG_EMBEDDING_API_TOKEN` and `EDGENTRAG_GENERATION_API_TOKEN` in this Studio's environment, or enter them at the hidden prompts. Keep the embedding token equal to the one configured in your local `.env`. Leave this Studio running while the local backend uses these URLs.

In [ ]:
from pathlib import Path
from getpass import getpass
import json, os, socket, subprocess, sys, time, urllib.error, urllib.request
from urllib.parse import urlsplit

if sys.version_info < (3, 12):
    raise RuntimeError('Use a Lightning Studio with Python 3.12 or newer.')
try:
    from lightning_sdk import Studio
except ImportError:
    raise RuntimeError('Lightning SDK is missing. Run: %pip install lightning-sdk') from None

PROJECT_DIR = Path(input('Absolute project root in this Studio (the folder containing app/backend): ').strip()).expanduser()
if not PROJECT_DIR.is_absolute(): raise ValueError('Enter an absolute project path.')
for service in ('embedding','generation'):
    if not (PROJECT_DIR / f'app/backend/src/edgentrag/{service}/app.py').is_file():
        raise RuntimeError(f'Missing {service} API under {PROJECT_DIR}.')
subprocess.run([sys.executable,'-m','pip','install','-e',f'{PROJECT_DIR}/app/backend[embedding,generation]'],check=True)

for service in ('embedding','generation'):
    key = f'EDGENTRAG_{service.upper()}_API_TOKEN'
    token = os.environ.get(key) or getpass(f'{key}: ')
    if not token or not token.strip(): raise RuntimeError(f'{key} cannot be empty.')
    os.environ[key] = token
os.environ['EDGENTRAG_EMBEDDING_DEVICE'] = 'cpu'
os.environ['EDGENTRAG_GENERATION_DEVICE'] = 'auto'
os.environ['EDGENTRAG_EMBEDDING_MODEL_NAME'] = 'sentence-transformers/all-MiniLM-L6-v2'
os.environ['EDGENTRAG_GENERATION_MODEL_NAME'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print('Project installed; both API tokens are available (values hidden).')

## Start and warm the APIs

The first authenticated call downloads and loads model weights, so it may take several minutes. Embeddings use CPU by default to leave GPU memory for generation.

In [ ]:
model_processes = globals().get('model_processes', {})
def stop_process(process):
    if process.poll() is None:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait(timeout=5)
def wait_health(url, process, log_path, attempts=60):
    for _ in range(attempts):
        if process.poll() is not None: raise RuntimeError(f'Process exited; inspect {log_path}.')
        try:
            with urllib.request.urlopen(url + '/health', timeout=5) as response: return json.load(response)
        except (urllib.error.URLError, TimeoutError): time.sleep(1)
    raise RuntimeError(f'Health check timed out; inspect {log_path}.')
def post_model(name, base_url):
    if name == 'embedding': route, payload = '/embed', {'texts':['A test document chunk.']}
    else: route, payload = '/generate', {'prompt':'Explain semantic search in one sentence.','max_new_tokens':64}
    request = urllib.request.Request(base_url + route, data=json.dumps(payload).encode(), headers={'Authorization':'Bearer '+os.environ[f'EDGENTRAG_{name.upper()}_API_TOKEN'],'Content-Type':'application/json'}, method='POST')
    with urllib.request.urlopen(request, timeout=600) as response: return json.load(response)

for name, port in {'embedding':8001,'generation':8003}.items():
    old = model_processes.get(name)
    if old is not None and old.poll() is None: raise RuntimeError(f'{name} already runs; clean up before rerunning.')
    with socket.socket() as probe:
        try: probe.bind(('127.0.0.1',port))
        except OSError: raise RuntimeError(f'Port {port} is occupied.') from None
started = []
try:
    for name, port in {'embedding':8001,'generation':8003}.items():
        log_path = Path(f'/tmp/edgentrag-{name}.log')
        env = os.environ.copy()
        env.pop(f"EDGENTRAG_{'GENERATION' if name == 'embedding' else 'EMBEDDING'}_API_TOKEN",None)
        with log_path.open('w') as log:
            process = subprocess.Popen([sys.executable,'-m','uvicorn',f'edgentrag.{name}.app:app','--host','0.0.0.0','--port',str(port)],cwd=PROJECT_DIR,env=env,stdout=log,stderr=subprocess.STDOUT)
        model_processes[name] = process; started.append(process)
        print(name, wait_health(f'http://127.0.0.1:{port}',process,log_path))
except Exception:
    for process in reversed(started): stop_process(process)
    raise

for name, port in {'embedding':8001,'generation':8003}.items():
    result = post_model(name,f'http://127.0.0.1:{port}')
    print(name,'ready:',result.get('model','response received'))

## Expose ports and test the public URLs

The ports must be reachable by your local API client. This calls Lightning's documented `Studio.add_ports()` method for each service, prints the SDK-provided URL, then makes an authenticated request through it. If the request fails, check that the Studio ports are exposed and that the processes above are still running. Avoid copying the editor's `web-ui?port=...` link.

In [ ]:
studio = Studio()
MODEL_URLS = {}
for name, port in {'embedding':8001,'generation':8003}.items():
    port_info = studio.add_ports(port)[0]
    urls = getattr(port_info,'urls',None)
    if not urls: raise RuntimeError(f'Lightning returned no exposed URL for port {port}.')
    url = urls[0].rstrip('/')
    parsed = urlsplit(url)
    if parsed.scheme != 'https' or not parsed.netloc or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError(f'Lightning did not return a clean public HTTPS URL for port {port}: {url!r}')
    MODEL_URLS[name] = url
    result = post_model(name,url)
    print(name,'public API verified:',result.get('model','response received'))

print('EDGENTRAG_USE_COLAB_FOR_EMBEDDING=false')
print('EDGENTRAG_USE_COLAB_FOR_LLM=false')
print('EDGENTRAG_LIGHTNING_EMBEDDING_SERVICE_URL=' + MODEL_URLS['embedding'])
print('EDGENTRAG_LIGHTNING_GENERATION_SERVICE_URL=' + MODEL_URLS['generation'])

## Configure the local backend

Copy the printed settings into your local `.env`, keep `EDGENTRAG_EMBEDDING_API_TOKEN` equal to the token entered above, and set the same generation token if you call generation directly. Restart the local API and worker after changing the embedding URL. Generation is not yet connected to search.

Optional cleanup: run the cell below with `STOP_SERVICES = True` to stop the model processes. This does not stop the Studio itself. The Studio must remain active while the URLs are in use.

In [ ]:
STOP_SERVICES = False
if STOP_SERVICES:
    for process in model_processes.values(): stop_process(process)
    MODEL_URLS.clear()
    print('Model APIs stopped.')
else:
    print('Services remain running.')